# 01 — EDA (Membre 2)

Analyse exploratoire de `reservation_annulee` et des variables associées.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils import (
    RANDOM_STATE, TARGET_COL, TIME_COL, RAW_DIR, PROCESSED_DIR, FIGURES_DIR,
    load_raw_data, set_seed, ensure_dirs, save_figure, PROJECT_ROOT,
)
set_seed(RANDOM_STATE)
ensure_dirs()
print("ROOT =", ROOT)
print("RANDOM_STATE =", RANDOM_STATE)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
train, test = load_raw_data()
print(train.shape, test.shape)
train.head()


In [ ]:
# Cible
rate = train[TARGET_COL].mean()
print(f"Taux global d'annulation: {rate:.4f}")
print(train[TARGET_COL].value_counts())
fig, ax = plt.subplots(figsize=(6,4))
train[TARGET_COL].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#2a9d8f","#e76f51"])
ax.set_title("Distribution de reservation_annulee")
save_figure(fig, "nb01_cible.png"); plt.show()


In [ ]:
# Manquants
miss = train.isna().sum()
print(miss[miss>0])
print("agent_id vides:", (train.agent_id.isna()|(train.agent_id.astype(str).str.strip()=="")).sum())


In [ ]:
# Temporalité
print("Train date_reservation:", train[TIME_COL].min().date(), "→", train[TIME_COL].max().date())
print("Test  date_reservation:", test[TIME_COL].min().date(), "→", test[TIME_COL].max().date())
tmp = train.copy(); tmp["mois"]=tmp[TIME_COL].dt.to_period("M").astype(str)
monthly = tmp.groupby("mois")[TARGET_COL].mean()
fig, ax = plt.subplots(figsize=(10,4))
monthly.plot(ax=ax, marker="o")
ax.set_title("Taux d'annulation mensuel")
save_figure(fig, "nb01_temporel.png"); plt.show()


In [ ]:
# Scénarios d'annulation
for col in ["tarif_remboursable","type_acompte","canal_reservation","client_type","type_destination"]:
    print("\n===", col, "===")
    print(train.groupby(col)[TARGET_COL].agg(["mean","count"]).sort_values("mean", ascending=False).round(3))
